In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import serial
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import os
from datetime import datetime


class ImagePacketDecoder:
    """
    RAW16 UART 图像包解析器（支持行扩展 + ADC）

    UART 包格式:
        FF FF +
        H * (W + 2) * u16   (图像 + 行扩展, BE) +
        1 * u16            (ADC, BE)

    npy 保存格式:
        shape = (H, W + 2)
        [0:W]   : 图像数据（Gray->Binary）
        [W:W+2] : 行扩展数据（Gray->Binary）

    ADC 单独返回（u16）
    """

    def __init__(self, width, height, calib_file=None):
        self.width = width
        self.height = height

        # 每行 u16 数（图像 + 行扩展）
        self.line_words = self.width + 2

        # 总 u16 数
        self.total_words = self.height * self.line_words + 1  # +1 ADC

        self.packet_size = 2 + self.total_words * 2

        self.calib = None
        if calib_file is not None:
            self._load_calibration(calib_file)

    def _load_calibration(self, calib_file):
        calib = np.load(calib_file)
        if calib.shape != (self.height, self.width):
            raise ValueError("Calibration shape mismatch")
        self.calib = calib.astype(np.int32)

    @staticmethod
    def gray_to_binary(img):
        # bin16 = img.copy().astype(np.uint16)
        # bin16 ^= bin16 >> 1
        # bin16 ^= bin16 >> 2
        # bin16 ^= bin16 >> 4
        # bin16 ^= bin16 >> 8
        # return bin16
        return img

    def feed_packet(self, packet):
        if len(packet) != self.packet_size:
            return None
        if packet[0] != 0xFF or packet[1] != 0xFF:
            return None

        data = packet[2:]
        buf8 = np.frombuffer(data, dtype=np.uint8)

        raw16 = (buf8[0::2].astype(np.uint16) << 8) | \
                buf8[1::2].astype(np.uint16)

        # 拆分 ADC
        adc_raw = raw16[-1]
        raw16_main = raw16[:-1]

        raw16_all = raw16_main.reshape((self.height, self.line_words))

        raw_img = raw16_all[:, :self.width]
        raw_ext = raw16_all[:, self.width:]

        img_bin = self.gray_to_binary(raw_img).astype(np.int32)
        if self.calib is not None:
            img_bin = img_bin - self.calib
            img_bin = np.clip(img_bin, 0, 65535)

        ext_bin = self.gray_to_binary(raw_ext).astype(np.uint16)

        return (
            img_bin.astype(np.uint16),  # (H, W)
            ext_bin,                    # (H, 2)
            int(adc_raw)                # u16 -> Python int
        )


def run_uart_capture(port, baud, width, height, calib_file=None):
    folder = datetime.now().strftime("capture_%Y%m%d_%H%M%S")
    os.makedirs(folder, exist_ok=True)

    npy_path = os.path.join(folder, "frame.npy")
    adc_path = os.path.join(folder, "adc.txt")

    decoder = ImagePacketDecoder(width, height, calib_file)

    buf = bytearray()

    try:
        with serial.Serial(port, baud, timeout=0.01) as ser:
            print("Receiving frames... Ctrl-C to stop")

            while True:
                buf.extend(ser.read(4096))

                while True:
                    idx = buf.find(b'\xFF\xFF')
                    if idx < 0:
                        buf = buf[-1:]
                        break

                    if len(buf) - idx < decoder.packet_size:
                        buf = buf[idx:]
                        break

                    packet = buf[idx:idx + decoder.packet_size]
                    buf = buf[idx + decoder.packet_size:]

                    result = decoder.feed_packet(packet)
                    if result is None:
                        continue

                    img_bin, ext_bin, adc_raw = result

                    frame_162 = np.concatenate([img_bin, ext_bin], axis=1)
                    np.save(npy_path, frame_162)

                    with open(adc_path, "a") as f:
                        f.write(f"{adc_raw}\n")

                    clear_output(wait=True)
                    plt.imshow(img_bin, cmap="gray")
                    plt.title(f"ADC raw = {adc_raw}")
                    plt.show()

    except KeyboardInterrupt:
        print("Stopped")


if __name__ == "__main__":
    run_uart_capture(
        port="COM31",
        baud=5_000_000,
        width=160,
        height=120,
        calib_file="capture_20260124_175116/raw_frames.npy",
    )


Stopped
